# Logistic Regression From Scratch: Chat Notes

These notes summarize and organize the ideas from the shared chat about a NumPy logistic regression script, pandas vs NumPy, binary cross-entropy, gradients, logits, train/test evaluation, and the next learning step into CIFAR-10 CNNs.

The core example is an admissions classifier:

- **Inputs / features**: `study_hours`, `practice_tests`
- **Target / label**: `admitted`
- **Goal**: predict whether `admitted` is `1` or `0`

In machine learning notation:

$$X = \begin{bmatrix}\text{study\_hours} & \text{practice\_tests}\end{bmatrix}$$

$$y = \text{admitted}$$


## 1. Why the Original Code Used NumPy Instead of Pandas

The original script loaded a CSV with NumPy:

```python
data = np.genfromtxt(DATA_PATH, delimiter=",", names=True)
```

That works because the dataset is small, simple, and numeric. The code immediately converts columns into NumPy arrays for math:

```python
X = np.column_stack([
    data["study_hours"].astype(float),
    data["practice_tests"].astype(float),
])
y = data["admitted"].astype(float).reshape(-1, 1)
```

Using NumPy is common in from-scratch ML tutorials because it keeps the lesson focused on vectors, matrices, and gradient descent.

But yes, pandas would also work, and it is often more readable for real-world data loading:

```python
data = pd.read_csv(DATA_PATH)
X = data[["study_hours", "practice_tests"]].to_numpy()
y = data["admitted"].to_numpy().reshape(-1, 1)
```

A good practical rule:

- Use **pandas** for loading, cleaning, inspecting, filtering, grouping, joining, and missing values.
- Use **NumPy arrays** for the model math once the data is ready.


In [1]:
# Mini example: generate admissions-like data so this notebook is self-contained.
import numpy as np

rng = np.random.default_rng(42)

n = 200
study_hours = rng.normal(loc=5.0, scale=2.0, size=n).clip(0, 10)
practice_tests = rng.poisson(lam=3.0, size=n).clip(0, 8)

# Hidden rule used to generate labels.
true_logit = 0.9 * study_hours + 0.8 * practice_tests - 6.0
true_probability = 1 / (1 + np.exp(-true_logit))
admitted = rng.binomial(1, true_probability)

X = np.column_stack([study_hours, practice_tests])
y = admitted.reshape(-1, 1).astype(float)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('First 5 rows of X:')
print(X[:5])
print('First 5 labels:')
print(y[:5].ravel())


X shape: (200, 2)
y shape: (200, 1)
First 5 rows of X:
[[5.60943416 0.        ]
 [2.92003179 1.        ]
 [6.50090239 2.        ]
 [6.88112943 4.        ]
 [1.09792962 2.        ]]
First 5 labels:
[0. 0. 1. 1. 0.]


In [2]:
# Optional pandas view, if pandas is installed.
try:
    import pandas as pd

    df = pd.DataFrame({
        'study_hours': study_hours,
        'practice_tests': practice_tests,
        'admitted': admitted,
    })
    display(df.head())
except ImportError:
    print('pandas is not installed in this environment. The NumPy examples still work.')


,study_hours,practice_tests,admitted
0,5.609434,0,0
1,2.920032,1,0
2,6.500902,2,1
3,6.881129,4,1
4,1.097930,2,0


## 2. `Path`, `__file__`, and `DATA_PATH`

The original code used:

```python
from pathlib import Path

DATA_PATH = Path(__file__).parent / "data" / "admissions_generated.csv"
```

Meaning:

- `Path` is Python's clean way to work with filesystem paths.
- `__file__` means the current script file.
- `.parent` means the folder containing that script.
- The `/` operator joins path pieces.

If the project looked like this:

```text
project/
├── train.py
└── data/
    └── admissions_generated.csv
```

then this expression points to:

```text
project/data/admissions_generated.csv
```

In a notebook, `__file__` usually is not defined, so notebooks often use:

```python
Path.cwd()
```

or a direct path relative to the notebook.


In [3]:
from pathlib import Path

notebook_folder = Path.cwd()
example_path = notebook_folder / 'data' / 'admissions_generated.csv'

print('Current folder:', notebook_folder)
print('Example data path:', example_path)


Current folder: /Users/garryyuan/machine_learning
Example data path: /Users/garryyuan/machine_learning/data/admissions_generated.csv


## 3. Sigmoid: Turning a Linear Score Into a Probability

The model first computes a linear score:

$$z = w_1x_1 + w_2x_2 + b$$

With two features, this is the same idea as:

$$ax_1 + bx_2 + c$$

In logistic regression, this linear score is called the **logit**.

Then the sigmoid function turns the logit into a probability:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Important behavior:

- Large positive `z` becomes a probability close to `1`.
- Large negative `z` becomes a probability close to `0`.
- `z = 0` becomes `0.5`.


In [4]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for z in [-6, -2, 0, 2, 6]:
    print(f'z={z:>3} -> sigmoid(z)={sigmoid(z):.4f}')


z= -6 -> sigmoid(z)=0.0025
z= -2 -> sigmoid(z)=0.1192
z=  0 -> sigmoid(z)=0.5000
z=  2 -> sigmoid(z)=0.8808
z=  6 -> sigmoid(z)=0.9975


## 4. What `admitted` Means

`admitted` is the **target variable**, also called the **label** or **output**.

Example dataset:

| study_hours | practice_tests | admitted |
|---:|---:|---:|
| 2 | 1 | 0 |
| 5 | 3 | 1 |
| 8 | 4 | 1 |

The model learns from:

```text
study_hours + practice_tests -> admitted
```

Usually:

```text
admitted = 1 means yes, admitted
admitted = 0 means no, not admitted
```


## 5. Building `X` and `y`

The feature matrix `X` contains input columns. In this project:

```python
X = [study_hours, practice_tests]
```

The target array `y` contains the correct answer:

```python
y = admitted
```

The `.reshape(-1, 1)` turns a flat array like this:

```python
[0, 1, 1, 0]
```

into a column vector like this:

```python
[[0],
 [1],
 [1],
 [0]]
```

That shape works cleanly with matrix multiplication and gradient calculations.


In [5]:
print('Feature matrix X shape:', X.shape)
print('Target vector y shape:', y.shape)
print('\nOne row means one student:')
print('study_hours, practice_tests -> admitted')
for i in range(5):
    print(f'{X[i, 0]:.1f}, {X[i, 1]:.0f} -> {int(y[i, 0])}')


Feature matrix X shape: (200, 2)
Target vector y shape: (200, 1)

One row means one student:
study_hours, practice_tests -> admitted
5.6, 0 -> 0
2.9, 1 -> 0
6.5, 2 -> 1
6.9, 4 -> 1
1.1, 2 -> 0


## 6. Train/Test Split and Feature Scaling

A train/test split lets us train on one part of the data and evaluate on unseen data.

Feature scaling helps gradient descent because the features are put on comparable scales. A common standardization is:

$$x_{scaled} = \frac{x - \mu}{\sigma}$$

where:

- $\mu$ is the training mean
- $\sigma$ is the training standard deviation

Use the training mean and standard deviation for both train and test data. Do not calculate scaling from the test set separately.


In [6]:
split_index = int(len(X) * 0.8)

X_train = X[:split_index]
y_train = y[:split_index]
X_test = X[split_index:]
y_test = y[split_index:]

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

print('X_train_scaled shape:', X_train_scaled.shape)
print('X_test_scaled shape:', X_test_scaled.shape)


X_train_scaled shape: (160, 2)
X_test_scaled shape: (40, 2)


## 7. Shape, Tuple Unpacking, Weights, Bias, Learning Rate, Epochs

This line defines two variables:

```python
n_samples, n_features = X_train_scaled.shape
```

If:

```python
X_train_scaled.shape == (160, 2)
```

then Python assigns:

```python
n_samples = 160
n_features = 2
```

That is called **tuple unpacking**.

Why use the shape? Because the model needs one weight per feature:

```python
weights = np.zeros((n_features, 1))
```

For two features, that gives:

$$w = \begin{bmatrix}0 \\ 0\end{bmatrix}$$

Other training setup values:

- `bias = 0.0`: starts the intercept at zero.
- `learning_rate = 0.1`: controls step size during gradient descent.
- `epochs = 3000`: how many times to loop over the training process.


In [7]:
n_samples, n_features = X_train_scaled.shape
weights = np.zeros((n_features, 1))
bias = 0.0
learning_rate = 0.1
epochs = 3000

print('n_samples:', n_samples)
print('n_features:', n_features)
print('weights shape:', weights.shape)
print('initial weights:')
print(weights)
print('initial bias:', bias)


n_samples: 160
n_features: 2
weights shape: (2, 1)
initial weights:
[[0.]
 [0.]]
initial bias: 0.0


## 8. Binary Cross-Entropy Loss

Binary cross-entropy measures how bad the model's predicted probabilities are for a binary classification task.

For one example:

- If the true label is `1`, the loss is low when the predicted probability is close to `1`.
- If the true label is `0`, the loss is low when the predicted probability is close to `0`.

Formula over the whole dataset:

$$L = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i\log(\hat y_i)+(1-y_i)\log(1-\hat y_i)\right]$$

Where:

- $y_i$ is the real label, either `0` or `1`.
- $\hat y_i$ is the predicted probability.
- $N$ is the number of training examples.

The tiny `epsilon` prevents taking `log(0)`, which is undefined.


In [8]:
def binary_cross_entropy(y_true, y_pred):
    epsilon = 1e-9
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(
        y_true * np.log(y_pred) +
        (1 - y_true) * np.log(1 - y_pred)
    )

examples = [
    (1, 0.99),
    (1, 0.50),
    (1, 0.10),
    (0, 0.01),
    (0, 0.50),
    (0, 0.90),
]

for actual, pred in examples:
    loss = binary_cross_entropy(np.array([[actual]]), np.array([[pred]]))
    print(f'y={actual}, predicted={pred:.2f}, loss={loss:.4f}')


y=1, predicted=0.99, loss=0.0101
y=1, predicted=0.50, loss=0.6931
y=1, predicted=0.10, loss=2.3026
y=0, predicted=0.01, loss=0.0101
y=0, predicted=0.50, loss=0.6931
y=0, predicted=0.90, loss=2.3026


## 9. Why the Formula Uses `1 / N`

The $\frac{1}{N}$ means we average the loss across all examples.

Without averaging, a dataset with 1000 rows would naturally have about 10 times more total loss than a dataset with 100 rows, even if the model quality were the same.

Averaging makes the loss easier to compare across different dataset sizes.

In code:

```python
np.mean(losses)
```

is the same idea as:

$$\frac{\text{sum of losses}}{\text{number of examples}}$$


In [9]:
losses_for_100_rows = np.full(100, 0.3)
losses_for_1000_rows = np.full(1000, 0.3)

print('Sum for 100 rows:', losses_for_100_rows.sum())
print('Sum for 1000 rows:', losses_for_1000_rows.sum())
print('Mean for 100 rows:', losses_for_100_rows.mean())
print('Mean for 1000 rows:', losses_for_1000_rows.mean())


Sum for 100 rows: 29.999999999999996
Sum for 1000 rows: 299.9999999999999
Mean for 100 rows: 0.3
Mean for 1000 rows: 0.2999999999999999


## 10. Training Loop: Gradient Descent

The training loop repeats this process many times:

1. Compute logits.
2. Convert logits to probabilities using sigmoid.
3. Measure loss.
4. Compute errors.
5. Compute gradients.
6. Update weights and bias.

Core code:

```python
logits = X_train_scaled @ weights + bias
probabilities = sigmoid(logits)
loss = binary_cross_entropy(y_train, probabilities)

errors = probabilities - y_train
d_weights = (X_train_scaled.T @ errors) / n_samples
d_bias = np.sum(errors) / n_samples

weights -= learning_rate * d_weights
bias -= learning_rate * d_bias
```

The `@` operator means matrix multiplication.

For one row with two features:

$$z = w_1x_1 + w_2x_2 + b$$

For every row at once:

$$z = Xw + b$$


In [10]:
# Reset model parameters.
n_samples, n_features = X_train_scaled.shape
weights = np.zeros((n_features, 1))
bias = 0.0
learning_rate = 0.1
epochs = 3000

history = []

for epoch in range(epochs):
    logits = X_train_scaled @ weights + bias
    probabilities = sigmoid(logits)
    loss = binary_cross_entropy(y_train, probabilities)

    errors = probabilities - y_train
    d_weights = (X_train_scaled.T @ errors) / n_samples
    d_bias = np.sum(errors) / n_samples

    weights -= learning_rate * d_weights
    bias -= learning_rate * d_bias

    if epoch % 300 == 0 or epoch == epochs - 1:
        train_predictions = (probabilities >= 0.5).astype(int)
        train_accuracy = np.mean(train_predictions == y_train)
        history.append((epoch, loss, train_accuracy))
        print(f'epoch {epoch:4d} | loss {loss:.4f} | train accuracy {train_accuracy:.2f}')

print('\nLearned weights:')
print(weights)
print('Learned bias:', bias)


epoch    0 | loss 0.6931 | train accuracy 0.59
epoch  300 | loss 0.4337 | train accuracy 0.82
epoch  600 | loss 0.4323 | train accuracy 0.82
epoch  900 | loss 0.4323 | train accuracy 0.82
epoch 1200 | loss 0.4323 | train accuracy 0.82
epoch 1500 | loss 0.4323 | train accuracy 0.82
epoch 1800 | loss 0.4323 | train accuracy 0.82
epoch 2100 | loss 0.4323 | train accuracy 0.82
epoch 2400 | loss 0.4323 | train accuracy 0.82
epoch 2700 | loss 0.4323 | train accuracy 0.82
epoch 2999 | loss 0.4323 | train accuracy 0.82

Learned weights:
[[1.33990514]
 [1.55802928]]
Learned bias: 0.721045698075645


## 11. Logit vs Probability

Your intuition was right: the logit is just the linear expression.

With two features:

$$z = ax_1 + bx_2 + c$$

In model notation:

$$z = w_1x_1 + w_2x_2 + b$$

The difference is that logistic regression has two stages:

```text
features -> linear score / logit -> sigmoid -> probability
```

So:

- **logit**: raw score, can be any real number.
- **probability**: sigmoid output, always between `0` and `1`.
- **prediction**: final class after applying a threshold, usually `0.5`.


In [11]:
student = np.array([[7.0, 4.0]])
student_scaled = (student - mean) / std

logit = student_scaled @ weights + bias
probability = sigmoid(logit)
prediction = (probability >= 0.5).astype(int)

print('Student features: study_hours=7, practice_tests=4')
print('Logit:', float(logit[0, 0]))
print('Probability admitted:', float(probability[0, 0]))
print('Prediction:', int(prediction[0, 0]))


Student features: study_hours=7, practice_tests=4
Logit: 3.4591714732769163
Probability admitted: 0.969503479641197
Prediction: 1


## 12. Evaluation on Test Data

After training, use the learned `weights` and `bias` on unseen test data:

```python
test_probabilities = sigmoid(X_test_scaled @ weights + bias)
test_predictions = (test_probabilities >= 0.5).astype(int)
test_accuracy = np.mean(test_predictions == y_test)
```

This tells us how well the model generalizes beyond the training data.

The threshold line:

```python
test_probabilities >= 0.5
```

means:

```text
probability >= 0.5 -> predict 1
probability < 0.5  -> predict 0
```


In [12]:
test_probabilities = sigmoid(X_test_scaled @ weights + bias)
test_predictions = (test_probabilities >= 0.5).astype(int)
test_accuracy = np.mean(test_predictions == y_test)

test_loss = binary_cross_entropy(y_test, test_probabilities)

print(f'test loss: {test_loss:.4f}')
print(f'test accuracy: {test_accuracy:.2f}')
print('\nFirst 10 test predictions:')
for i in range(10):
    print(
        f'prob={test_probabilities[i, 0]:.3f} '
        f'pred={test_predictions[i, 0]} '
        f'actual={int(y_test[i, 0])}'
    )


test loss: 0.3267
test accuracy: 0.78

First 10 test predictions:
prob=0.592 pred=1 actual=0
prob=0.454 pred=0 actual=1
prob=0.998 pred=1 actual=1
prob=0.645 pred=1 actual=0
prob=0.907 pred=1 actual=1
prob=0.386 pred=0 actual=0
prob=0.815 pred=1 actual=1
prob=0.987 pred=1 actual=1
prob=0.929 pred=1 actual=1
prob=0.861 pred=1 actual=1


## 13. Full Compact Version

Here is the whole logistic regression flow in one compact cell.


In [13]:
import numpy as np

rng = np.random.default_rng(7)
n = 300
study_hours = rng.normal(5, 2, n).clip(0, 10)
practice_tests = rng.poisson(3, n).clip(0, 8)
logit_rule = 0.9 * study_hours + 0.8 * practice_tests - 6
prob_rule = 1 / (1 + np.exp(-logit_rule))
admitted = rng.binomial(1, prob_rule)

X = np.column_stack([study_hours, practice_tests])
y = admitted.reshape(-1, 1).astype(float)

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def binary_cross_entropy(y_true, y_pred):
    epsilon = 1e-9
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

n_samples, n_features = X_train_scaled.shape
weights = np.zeros((n_features, 1))
bias = 0.0
learning_rate = 0.1
epochs = 3000

for epoch in range(epochs):
    logits = X_train_scaled @ weights + bias
    probabilities = sigmoid(logits)
    errors = probabilities - y_train

    d_weights = (X_train_scaled.T @ errors) / n_samples
    d_bias = np.sum(errors) / n_samples

    weights -= learning_rate * d_weights
    bias -= learning_rate * d_bias

train_accuracy = np.mean((sigmoid(X_train_scaled @ weights + bias) >= 0.5) == y_train)
test_accuracy = np.mean((sigmoid(X_test_scaled @ weights + bias) >= 0.5) == y_test)

print(f'train accuracy: {train_accuracy:.2f}')
print(f'test accuracy: {test_accuracy:.2f}')
print('weights:', weights.ravel())
print('bias:', bias)


train accuracy: 0.82
test accuracy: 0.75
weights: [1.98111034 1.59644671]
bias: 0.8345313858722052


## 14. CIFAR-10 CNN: The Next Step After MNIST

The final part of the chat asked about this recommendation:

> Start with CIFAR-10 CNN after basic MNIST.

That means: once you understand a basic MNIST neural network, the next useful project is image classification on CIFAR-10 using a convolutional neural network.

### What is CIFAR-10?

CIFAR-10 is a classic image classification dataset with:

- 60,000 color images
- 10 classes
- 32 by 32 pixel images

Classes include:

```text
airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
```

### Why is it harder than MNIST?

MNIST is simple:

- grayscale digits
- centered objects
- simple shapes

CIFAR-10 is harder:

- color images
- real-world objects
- backgrounds
- different positions and textures

### What CIFAR-10 teaches

A CIFAR-10 CNN project teaches:

- `torchvision.datasets`
- data augmentation
- convolution layers: `Conv2d`
- pooling: `MaxPool2d`
- normalization: `BatchNorm2d`
- better training loops
- architecture experiments

A natural learning path is:

1. Logistic regression from scratch with NumPy.
2. Basic neural network on MNIST.
3. CNN on CIFAR-10.
4. Experiment with deeper CNNs, augmentation, schedulers, and regularization.


## 15. Quick Glossary

| Term | Meaning |
|---|---|
| Feature | Input column used for prediction, such as `study_hours`. |
| Target / label | Correct answer the model learns to predict, such as `admitted`. |
| `X` | Feature matrix. Rows are examples, columns are features. |
| `y` | Target values. Usually shaped as a column vector. |
| Weight | Learned coefficient for a feature. |
| Bias | Learned intercept added to the linear score. |
| Logit | Raw linear score: `X @ weights + bias`. |
| Sigmoid | Function that converts logits into probabilities. |
| Probability | Model confidence between `0` and `1`. |
| Prediction | Final class after thresholding probability. |
| BCE loss | Binary cross-entropy, used for binary classification. |
| Gradient | Direction showing how to change parameters to reduce loss. |
| Learning rate | Step size for parameter updates. |
| Epoch | One training iteration cycle in this simple loop. |
| Test set | Data held out to evaluate generalization. |


## 16. Key Takeaways

- The original code did not need pandas because NumPy can load simple numeric CSVs and perform all the matrix math.
- Pandas is still a great choice for cleaner data loading and inspection.
- `n_samples, n_features = X.shape` defines two variables through tuple unpacking.
- Logistic regression computes a linear score first: $z = w_1x_1 + w_2x_2 + b$.
- That score is called the logit.
- Sigmoid converts the logit into a probability.
- Binary cross-entropy rewards confident correct probabilities and punishes confident wrong probabilities.
- The $1/N$ average keeps the loss comparable across dataset sizes.
- Gradient descent updates `weights` and `bias` a little bit each epoch.
- Test accuracy checks whether the model works on data it did not train on.
- After MNIST, CIFAR-10 CNNs are a strong next step because they introduce color image classification and convolutional layers.
